# Model 7 — 2-Stage Recommendation Engine

Ranks all portal candidates for a given program, producing a **Top 50** pool (Stage 1) and refining it down to a personalized **Top 10** (Stage 2).

| Stage | Input | Method | Output |
|---|---|---|---|
| **Stage 1** | Full candidate pool | Availability gate → vectorized rank score | Top 50 |
| **Stage 2** | Top 50 | User-preference reweighting + risk-tolerance penalty | Final Top 10 |

**Inputs required in `df_candidates`:**

| Column | Type | Description |
|---|---|---|
| `player_id` | int | Unique player identifier |
| `player_name` | str | Display name |
| `availability_status` | str | `'available'` or `'committed'` |
| `player_projection` | float | Baseline RAPM-like talent score |
| `data_confidence` | float | 0.0–1.0 sample-size confidence multiplier |
| `team_rating_delta` | float | Projected efficiency lift for destination program |
| `scheme_fit` | float | Scheme fit score [0–100] from Model 3 |
| `gap_match` | float | Gap match score [0–100] from Model 4 |



>
 **`data_confidence`** is not yet in production data. Synthetic values (0.40–1.0) are used here; replace with real per-player confidence scores when available (e.g., derived from `games_played` / `minutes` thresholds).

In [4]:
import numpy as np
import pandas as pd
from typing import Optional
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Libraries loaded')

Libraries loaded


## Shared Helper: Overall Fit Score

Collapses the available fit sub-scores into a single `overall_fit` value in **[0, 100]** using a modular weight dictionary.

**Current weights** (split evenly between the two live models):
```
overall_fit = 0.5 × scheme_fit + 0.5 × gap_match
```

**To add `role_fit` / `program_fit` when Models 5–6 are merged:**
1. Ensure those columns exist in `df_candidates`
2. Uncomment the keys in `DEFAULT_FIT_WEIGHTS` and re-proportion so weights sum to 1.0
3. Mirror the same keys in `user_preferences` inside `refine_to_top_10()`

In [5]:
DEFAULT_FIT_WEIGHTS: dict = {
    'scheme_fit': 0.5,
    'gap_match':  0.5,
    # 'role_fit':    0.0,   # placeholder — wire in when Model 5 (role scorer) is merged
    # 'program_fit': 0.0,   # placeholder — wire in when Model 6 (program fit) is merged
}


def calculate_overall_fit(df: pd.DataFrame, weights: Optional[dict] = None) -> pd.Series:
    """Weighted sum of fit sub-scores. Output stays in [0, 100]."""
    w = weights if weights is not None else DEFAULT_FIT_WEIGHTS
    total = sum(w.values())
    if abs(total - 1.0) > 1e-6:
        raise ValueError(f'Fit weights must sum to 1.0, got {total:.6f}')
    return sum(df[col] * wt for col, wt in w.items())


print('calculate_overall_fit() ready')
print(f'Default weights: {DEFAULT_FIT_WEIGHTS}')

calculate_overall_fit() ready
Default weights: {'scheme_fit': 0.5, 'gap_match': 0.5}


## Stage 1: Hard Gate & Vectorized Top-50

Filters the full candidate pool down to the **50 highest-ranked available players** using three additive signals:

```
adjusted_projection = player_projection × data_confidence
stage1_rank_score   = adjusted_projection + team_rating_delta + (overall_fit / 100)
```

**Why divide `overall_fit` by 100?**  
`scheme_fit` and `gap_match` are on a [0, 100] scale, while `player_projection` and `team_rating_delta` are RAPM-scale values (roughly –5 to +12). Dividing by 100 maps fit to [0, 1], keeping it a proportional contribution rather than a dominating one.

**Why multiply by `data_confidence`?**  
Players with thin sample history (few games, limited minutes) get a discounted projection. When `data_confidence == 1.0` the multiplication is a no-op; at `0.5` it halves the projection.

In [6]:
def generate_top_50_candidates(
    df: pd.DataFrame,
    weights: Optional[dict] = None,
    filter_available: bool = True,
) -> pd.DataFrame:
    """Availability gate → vectorized rank score → Top-50 candidate pool.

    Parameters
    ----------
    df               : df_candidates with required columns (see notebook header).
    weights          : Fit sub-score weights. Defaults to DEFAULT_FIT_WEIGHTS.
    filter_available : Hard-gates to availability_status == 'available' when True.
                       Set False to include committed/pending players for planning.

    Returns
    -------
    DataFrame of <= 50 rows sorted descending by stage1_rank_score,
    with overall_fit, adjusted_projection, and stage1_rank_score appended.
    """
    working = df.copy()

    # Hard gate: drop unavailable players
    if filter_available:
        working = working[working['availability_status'] == 'available'].copy()

    working['overall_fit'] = calculate_overall_fit(working, weights)

    working['adjusted_projection'] = (
        working['player_projection'] * working['data_confidence']
    )

    working['stage1_rank_score'] = (
        working['adjusted_projection']
        + working['team_rating_delta']
        + (working['overall_fit'] / 100.0)
    )

    top50 = (
        working
        .sort_values('stage1_rank_score', ascending=False)
        .head(50)
        .reset_index(drop=True)
    )

    print('── Stage 1: Top-50 Generation ─────────────────────────────────────')
    print(f'  Input pool       : {len(df):,} players')
    if filter_available:
        n_avail = (df['availability_status'] == 'available').sum()
        print(f'  After gate       : {n_avail:,} available  ({len(df) - n_avail} filtered out)')
    print(f'  Rank score range : {top50["stage1_rank_score"].min():.3f} → '
          f'{top50["stage1_rank_score"].max():.3f}')
    print(f'  Fit score range  : {top50["overall_fit"].min():.1f} → '
          f'{top50["overall_fit"].max():.1f}')
    print(f'  Avg data confid. : {top50["data_confidence"].mean():.3f}')
    print()
    print(top50[[
        'player_name', 'player_projection', 'data_confidence',
        'adjusted_projection', 'team_rating_delta',
        'overall_fit', 'stage1_rank_score',
    ]].head(10).to_string(index=False))
    return top50


print('generate_top_50_candidates() ready')

generate_top_50_candidates() ready


## Stage 2: User Preference Refinement → Top 10

Takes the Top-50 pool and re-ranks it using two personalization levers:

**1. User preference weights**  
The recruiter can shift the balance between `scheme_fit` and `gap_match` (and, in the future, `role_fit` / `program_fit`). Weights are auto-normalized, so only the *ratio* matters — passing `scheme_fit=2, gap_match=1` is the same as `0.67 / 0.33`.

**2. Risk tolerance**  
Controls how much uncertainty is tolerated in a player's data history:

| Setting | Confidence floor | Penalty rate | Effect |
|---|---|---|---|
| `low` | 0.70 | 2.0× | Aggressively penalizes thin-data players |
| `medium` | 0.50 | 1.0× | Mild penalty below threshold (default) |
| `high` | none | 0× | No penalty; raw talent rewarded |

The penalty formula proxies the **Transfer Success Model's efficiency regression** — players with limited sample history are discounted to reflect prediction uncertainty:
```
confidence_penalty = max(0, floor − data_confidence) × penalty_rate
final_rec_score    = adjusted_projection + team_rating_delta
                   + (personalized_fit / 100) − confidence_penalty
```

In [7]:
_RISK_CONFIG: dict = {
    'low':    {'confidence_floor': 0.70, 'penalty': 2.0},
    'medium': {'confidence_floor': 0.50, 'penalty': 1.0},
    'high':   {'confidence_floor': 0.00, 'penalty': 0.0},
}


def refine_to_top_10(
    df_top_50: pd.DataFrame,
    user_preferences: Optional[dict] = None,
    risk_tolerance: str = 'medium',
) -> pd.DataFrame:
    """Re-rank Top-50 with personalized fit weights and risk penalty → Top 10.

    Parameters
    ----------
    df_top_50        : Direct output of generate_top_50_candidates().
    user_preferences : Dict with optional keys:
                         'scheme_fit_weight' (float, default 0.5)
                         'gap_match_weight'  (float, default 0.5)
                       Weights are auto-normalized — only ratios matter.
                       Future keys: 'role_fit_weight', 'program_fit_weight'.
    risk_tolerance   : 'low' | 'medium' | 'high' — see _RISK_CONFIG above.

    Returns
    -------
    DataFrame of <= 10 rows sorted by final_rec_score desc.
    All Stage-1 columns are retained plus:
      final_rank, personalized_fit, confidence_penalty, final_rec_score.
    """
    if risk_tolerance not in _RISK_CONFIG:
        raise ValueError(
            f"risk_tolerance must be one of {list(_RISK_CONFIG)}, got '{risk_tolerance}'"
        )

    prefs = user_preferences or {}

    # Build and normalize user-defined fit weights
    raw_w = {
        'scheme_fit': prefs.get('scheme_fit_weight', 0.5),
        'gap_match':  prefs.get('gap_match_weight',  0.5),
        # 'role_fit':    prefs.get('role_fit_weight',    0.0),   # future
        # 'program_fit': prefs.get('program_fit_weight', 0.0),   # future
    }
    total_w = sum(raw_w.values())
    if total_w == 0:
        raise ValueError('All preference weights are zero — cannot normalize.')
    norm_w = {k: v / total_w for k, v in raw_w.items()}

    working = df_top_50.copy()

    working['personalized_fit'] = calculate_overall_fit(working, norm_w)

    cfg       = _RISK_CONFIG[risk_tolerance]
    shortfall = (cfg['confidence_floor'] - working['data_confidence']).clip(lower=0.0)
    working['confidence_penalty'] = shortfall * cfg['penalty']

    working['final_rec_score'] = (
        working['adjusted_projection']
        + working['team_rating_delta']
        + (working['personalized_fit'] / 100.0)
        - working['confidence_penalty']
    )

    top10 = (
        working
        .sort_values('final_rec_score', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    top10.insert(0, 'final_rank', range(1, len(top10) + 1))

    print('── Stage 2: Top-10 Refinement ─────────────────────────────────────')
    print(f'  Risk tolerance      : {risk_tolerance}')
    print(f'  Confidence floor    : {cfg["confidence_floor"]}  |  Penalty rate : {cfg["penalty"]}x')
    print(f'  Personalized weights: {norm_w}')
    print(f'  Final score range   : {top10["final_rec_score"].min():.3f} → '
          f'{top10["final_rec_score"].max():.3f}')
    return top10


print('refine_to_top_10() ready')

refine_to_top_10() ready


## Sample Verification

Builds a **mock candidate pool of 120 players** to verify the full pipeline end-to-end.

Synthetic distributions used:

| Field | Distribution | Notes |
|---|---|---|
| `player_projection` | Uniform(−5, 12) | RAPM-like talent range |
| `data_confidence` | Clip(Normal(0.72, 0.15), 0.40, 1.0) | Anchored toward high confidence |
| `team_rating_delta` | Uniform(−3, 5) | Efficiency impact range |
| `scheme_fit` | Clip(Normal(55, 20), 0, 100) | Slightly above average pool |
| `gap_match` | Clip(Normal(50, 22), 0, 100) | Centered at midpoint |
| `availability_status` | Bernoulli(p=0.75 available) | 75% of pool in portal |

The example below runs Stage 1 with default weights, then Stage 2 with a user preference favoring scheme fit (`0.65`) over gap match (`0.35`) and `risk_tolerance='low'` to penalize thin-data players.

In [8]:
_rng = np.random.default_rng(99)
_N   = 120

df_candidates = pd.DataFrame({
    'player_id':           range(1, _N + 1),
    'player_name':         [f'Player_{i:03d}' for i in range(1, _N + 1)],
    'position':            _rng.choice(['PG', 'SG', 'SF', 'PF', 'C'], size=_N),
    'availability_status': _rng.choice(
        ['available', 'committed'], size=_N, p=[0.75, 0.25]
    ),
    'player_projection':   _rng.uniform(-5.0, 12.0, size=_N).round(3),
    'data_confidence':     np.clip(_rng.normal(0.72, 0.15, size=_N), 0.40, 1.0).round(3),
    'team_rating_delta':   _rng.uniform(-3.0, 5.0, size=_N).round(3),
    'scheme_fit':          np.clip(_rng.normal(55, 20, size=_N), 0, 100).round(1),
    'gap_match':           np.clip(_rng.normal(50, 22, size=_N), 0, 100).round(1),
    # Future columns — uncomment when Models 5-6 are integrated:
    # 'role_fit':    np.clip(_rng.normal(52, 18, size=_N), 0, 100).round(1),
    # 'program_fit': np.clip(_rng.normal(48, 20, size=_N), 0, 100).round(1),
})

print(f'Mock candidate pool  : {len(df_candidates)} players')
print(f'  Available          : {(df_candidates["availability_status"] == "available").sum()}')
print(f'  Committed          : {(df_candidates["availability_status"] == "committed").sum()}')
print(f'  Positions          : {df_candidates["position"].value_counts().to_dict()}')
print(f'  Avg data_confidence: {df_candidates["data_confidence"].mean():.3f}')
df_candidates.head()

Mock candidate pool  : 120 players
  Available          : 87
  Committed          : 33
  Positions          : {'SG': 31, 'PF': 30, 'SF': 24, 'C': 18, 'PG': 17}
  Avg data_confidence: 0.698


,player_id,player_name,position,availability_status,player_projection,data_confidence,team_rating_delta,scheme_fit,gap_match
0,1,Player_001,C,available,-0.007,0.583,-0.607,38.5,61.7
1,2,Player_002,SF,available,8.445,0.725,3.375,67.6,67.8
2,3,Player_003,PF,available,6.966,0.400,4.714,31.5,59.8
3,4,Player_004,SF,available,5.131,0.647,-1.390,75.4,8.3
4,5,Player_005,PG,available,8.476,0.715,-1.282,61.2,31.4


### Run Stage 1 → Stage 2

In [9]:
# Stage 1 — default fit weights, available players only
df_top_50 = generate_top_50_candidates(
    df=df_candidates,
    weights=None,
    filter_available=True,
)

print()
print('─' * 70)
print()

# Stage 2 — scheme fit preferred; conservative on data confidence
user_preferences = {
    'scheme_fit_weight': 0.65,
    'gap_match_weight':  0.35,
}

final_top10 = refine_to_top_10(
    df_top_50=df_top_50,
    user_preferences=user_preferences,
    risk_tolerance='low',
)

── Stage 1: Top-50 Generation ─────────────────────────────────────
  Input pool       : 120 players
  After gate       : 87 available  (33 filtered out)
  Rank score range : 4.071 → 14.492
  Fit score range  : 29.1 → 88.6
  Avg data confid. : 0.702

player_name  player_projection  data_confidence  adjusted_projection  team_rating_delta  overall_fit  stage1_rank_score
 Player_050             11.426            0.774             8.843724              4.954        69.45          14.492224
 Player_059             11.557            0.688             7.951216              4.822        41.10          13.184216
 Player_100             11.950            0.797             9.524150              2.727        57.95          12.830650
 Player_092              8.983            0.796             7.150468              4.439        50.90          12.098468
 Player_108              9.452            0.755             7.136260              4.501        42.35          12.060760
 Player_115             11.58

### Final Top-10 — All Component Scores

The table below retains every constituent score so each recommendation is fully explainable:

| Column | Meaning |
|---|---|
| `player_projection` | Raw RAPM-like talent score |
| `data_confidence` | Sample-size confidence (green = high, red = low) |
| `team_rating_delta` | Efficiency contribution to destination program |
| `overall_fit` | Stage-1 fit score (default equal weights) |
| `personalized_fit` | Stage-2 fit score (user-reweighted) |
| `adjusted_projection` | `player_projection × data_confidence` |
| `confidence_penalty` | Score deduction from risk tolerance check |
| `stage1_rank_score` | Pre-refinement rank score |
| `final_rec_score` | Final personalized score used for Top-10 ranking |

In [10]:
DISPLAY_COLS = [
    'final_rank', 'player_name', 'position', 'availability_status',
    'player_projection', 'data_confidence', 'team_rating_delta',
    'scheme_fit', 'gap_match',
    'overall_fit', 'personalized_fit',
    'adjusted_projection', 'confidence_penalty',
    'stage1_rank_score', 'final_rec_score',
]

FMT = {
    'player_projection':   '{:.3f}',
    'data_confidence':     '{:.3f}',
    'team_rating_delta':   '{:.3f}',
    'scheme_fit':          '{:.1f}',
    'gap_match':           '{:.1f}',
    'overall_fit':         '{:.1f}',
    'personalized_fit':    '{:.1f}',
    'adjusted_projection': '{:.3f}',
    'confidence_penalty':  '{:.3f}',
    'stage1_rank_score':   '{:.3f}',
    'final_rec_score':     '{:.3f}',
}

try:
    styled = (
        final_top10[DISPLAY_COLS]
        .style
        .format(FMT)
        .background_gradient(subset=['final_rec_score'], cmap='YlGn')
        .background_gradient(subset=['data_confidence'],  cmap='RdYlGn')
        .background_gradient(subset=['confidence_penalty'], cmap='Reds_r')
        .set_caption(
            "Final Top-10 | risk_tolerance='low' | scheme_fit x0.65 / gap_match x0.35"
        )
    )
    display(styled)
except Exception:
    print(final_top10[DISPLAY_COLS].to_string(index=False))

,final_rank,player_name,position,availability_status,player_projection,data_confidence,team_rating_delta,scheme_fit,gap_match,overall_fit,personalized_fit,adjusted_projection,confidence_penalty,stage1_rank_score,final_rec_score
0,1,Player_050,SF,available,11.426,0.774,4.954,74.8,64.1,69.4,71.1,8.844,0.000,14.492,14.508
1,2,Player_059,PG,available,11.557,0.688,4.822,24.4,57.8,41.1,36.1,7.951,0.024,13.184,13.110
2,3,Player_100,PF,available,11.950,0.797,2.727,63.0,52.9,58.0,59.5,9.524,0.000,12.831,12.846
3,4,Player_092,PF,available,8.983,0.796,4.439,62.2,39.6,50.9,54.3,7.150,0.000,12.098,12.132
4,5,Player_108,SF,available,9.452,0.755,4.501,49.1,35.6,42.4,44.4,7.136,0.000,12.061,12.081
5,6,Player_115,SG,available,11.585,0.707,3.112,53.8,6.8,30.3,37.4,8.191,0.000,11.606,11.676
6,7,Player_072,SG,available,11.898,0.555,4.298,40.4,71.8,56.1,51.4,6.603,0.290,11.462,11.125
7,8,Player_068,PG,available,8.529,0.750,3.863,30.5,48.7,39.6,36.9,6.397,0.000,10.656,10.628
8,9,Player_117,PG,available,9.581,0.692,3.123,55.7,30.2,43.0,46.8,6.630,0.016,10.183,10.205
9,10,Player_002,SF,available,8.445,0.725,3.375,67.6,67.8,67.7,67.7,6.123,0.000,10.175,10.174
